# Model Heuristics and Rankings 

This notebook describes features from our merged_df considered and how they are prioritised in our ranking system. 

Simple weighted heuristics based system rather than model.... happy to accept other ideas and hear challenges on this. 

Assumptions: 
- BNPL company gets payed a percentage of each transaction and late fees from consumers (lots of revenue, lots of customers) 
- Main goal for the company is profit and growth 

Considerations: 
- fraud treated as a 'penalty' not an outright exclusion 
- diverse categories considered to maintain robust merchant profile 
- data from 2021 - 2022 

## Profit target 
Expected profit can be defined as revenue * take rate * (1 - fraud_probability)

In [1]:
from pyspark.sql import SparkSession, functions as F

def get_spark_session(app_name: str = "SparkSession") -> SparkSession:
    """
    Create a SparkSession with the given app name.

    """
    return SparkSession.builder.appName(app_name).getOrCreate()

spark = get_spark_session()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/04 13:25:09 WARN Utils: Your hostname, CompuPau, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/04 13:25:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/pau_l/project-2/venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/04 13:25:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/04 13:25:11 WARN Utils: Service 'SparkUI' could not

In [4]:
# get profit target from merged_df and group by category. 

#load merged_df from parquet file
merged_df = spark.read.parquet("../data/curated/merged_transactions_sa2.parquet")

#add column for profit value based on formula revenue * take rate * (1-fraud probability)
merged_df = merged_df.withColumn("profit", F.col("dollar_value") * (F.col("take_rate") / 100) * (1 - F.col("fraud_probability")))

maybe growth score?? 
(2022 - 2021 revenue) / 2021 revenue

other: customer growth, recent growth (3-6 months)

In [ ]:
# calculate growth score as per the above formula for each merchant and group by category.

## Customer Profile - Late Fee Potential 

Other potential ideas: unique customer count, rate of repeat customers, concentration of spending 
other: how many unique postcodes served, mix of incomes 

In [9]:
# determine high, medium, low and very low income postcodes based on the distribution of income in merged_df.
# high proportion of low- or middle-income customers may indicate higher late fee potential 

#check the distribution of income in merged_df
merged_df.select("Median_tot_fam_inc_weekly").describe().show(vertical=True)

#split into quartiles and assign labels
q1, q2, q3 = merged_df.approxQuantile("Median_tot_fam_inc_weekly", [0.25, 0.5, 0.75], 0.01)

#split into quartiles and assign labels
merged_df = merged_df.withColumn("income_category", F.when(F.col("Median_tot_fam_inc_weekly").isNull(), None)
                            .when(F.col("Median_tot_fam_inc_weekly") < q1, "very_low")
                            .when(F.col("Median_tot_fam_inc_weekly") < q2, "low")
                            .when(F.col("Median_tot_fam_inc_weekly") < q3, "medium")
                            .otherwise("high"))

# calculate proportion of customers in each income category for each merchant and group by category.
income_proportions = merged_df.groupBy("merchant_abn", "income_category").count()


-RECORD 0-----------------------------------------
 summary                   | count                
 Median_tot_fam_inc_weekly | 11366510             
-RECORD 1-----------------------------------------
 summary                   | mean                 
 Median_tot_fam_inc_weekly | 1970.247075802266    
-RECORD 2-----------------------------------------
 summary                   | stddev               
 Median_tot_fam_inc_weekly | 605.0958899240484    
-RECORD 3-----------------------------------------
 summary                   | min                  
 Median_tot_fam_inc_weekly | 5.214800105728080... 
-RECORD 4-----------------------------------------
 summary                   | max                  
 Median_tot_fam_inc_weekly | 6999.999863758258    



## Merchant Stability 
how stable is merchant revenue each month 

In [ ]:
# determine monthly income per merchant and find variance across 2021-2022

## Fraud trend 

is fraud random? or does it always concentrate? is it increasing/decreasing

## Performance according to overall market 

how does a merchant perform related to interest rates rising and falling (stay strong, follow, fall off with high interest rates) and does this map to is_essential or not 

# Combine into a final Score 

split to be decided.... 

overall rank approx: 
- profitability
- customer profile
- stability and performance in terms of market?? 


## Diversity 

pick top 10 per category 

## Final Conditions 

- minimum activity 
- reviews 
- human explanation (transparency) 